<a href="https://colab.research.google.com/github/rodrigoeduardo9/LosLederesSemana7/blob/main/LosLederesParcial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Proyecto de Predicción de Riesgo de Deserción Universitaria con Regresión Múltiple

In [1]:
# Importar librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error

# Establecer estilo para gráficos
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

print("Librerías importadas correctamente.")

Librerías importadas correctamente.


## 1. Carga de Datos

En esta sección, cargaremos el dataset proporcionado por el usuario en formato CSV y realizaremos una exploración inicial para entender su estructura, contenido y propiedades estadísticas. Es crucial que el archivo CSV esté subido en el entorno de Google Colab para que esta celda funcione correctamente.

### Cargar el Dataset desde CSV

**Instrucciones:**
1.  Haz clic en el icono de la carpeta en el panel lateral izquierdo de Colab.
2.  Haz clic en el icono de 'Cargar' (Upload) y sube tu archivo CSV. Puedes nombrarlo `dropout_data.csv` o ajustar el nombre en el código a continuación.
3.  Ejecuta la celda de código para cargar los datos en un DataFrame de pandas.

In [2]:
# Cargar el dataset
try:
    # Se asume que el archivo CSV se llama 'dropout_data.csv'
    # Si tu archivo tiene otro nombre, por favor actualiza esta línea.
    df = pd.read_csv('dataset.csv', sep=';') # <-- Corregido para usar ';' como separador
    print("Dataset 'dropout_data.csv' cargado exitosamente.")
except FileNotFoundError:
    print("Error: El archivo 'dropout_data.csv' no fue encontrado. Asegúrate de haberlo subido al entorno de Colab.")
    print("Por favor, sube tu archivo CSV o cambia el nombre del archivo en el código.")
    df = None # Asegurarse de que df no esté definido si hay un error

if df is not None:
    print("\n--- Primeras 5 filas del dataset (df.head()) ---")
    display(df.head())

Dataset 'dropout_data.csv' cargado exitosamente.

--- Primeras 5 filas del dataset (df.head()) ---


,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.0,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666.666.666.666.600,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.0,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.4,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.0,0,13.9,-0.3,0.79,Graduate


### Información General del DataFrame (df.info())

La función `df.info()` proporciona un resumen conciso del DataFrame, incluyendo el número de entradas, el número de columnas, los tipos de datos de cada columna y la cantidad de valores no nulos. Esta información es vital para identificar si existen valores faltantes o si los tipos de datos son coherentes con lo esperado para cada variable.

In [3]:
if df is not None:
    print("\n--- Información del DataFrame (df.info()) ---")
    df.info()


--- Información del DataFrame (df.info()) ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance	                     4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  

### Estadísticas Descriptivas (df.describe())

`df.describe()` genera estadísticas descriptivas para las columnas numéricas, como la media, desviación estándar, valores mínimos y máximos, y los cuartiles. Esto nos da una idea rápida de la distribución y rango de los datos numéricos, ayudando a detectar posibles *outliers* o inconsistencias.

In [4]:
if df is not None:
    print("\n--- Estadísticas descriptivas del DataFrame (df.describe()) ---")
    display(df.describe())


--- Estadísticas descriptivas del DataFrame (df.describe()) ---


,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 1st sem (approved),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP
count,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,...,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000
mean,1.178571,18.669078,1.727848,8856.642631,0.890823,4.577758,132.613314,1.873192,19.561935,22.275316,...,4.706600,0.137658,0.541817,6.232143,8.063291,4.435805,0.150316,11.566139,1.228029,0.001969
std,0.605747,17.484682,1.313793,2063.566416,0.311897,10.216592,13.188332,6.914514,15.603186,15.343108,...,3.094238,0.690880,1.918546,2.195951,3.947951,3.014764,0.753774,2.663850,1.382711,2.269935
min,1.000000,1.000000,0.000000,33.000000,0.000000,1.000000,95.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.600000,-0.800000,-4.060000
25%,1.000000,1.000000,1.000000,9085.000000,1.000000,1.000000,125.000000,1.000000,2.000000,3.000000,...,3.000000,0.000000,0.000000,5.000000,6.000000,2.000000,0.000000,9.400000,0.300000,-1.700000
50%,1.000000,17.000000,1.000000,9238.000000,1.000000,1.000000,133.100000,1.000000,19.000000,19.000000,...,5.000000,0.000000,0.000000,6.000000,8.000000,5.000000,0.000000,11.100000,1.400000,0.320000
75%,1.000000,39.000000,2.000000,9556.000000,1.000000,1.000000,140.000000,1.000000,37.000000,37.000000,...,6.000000,0.000000,0.000000,7.000000,10.000000,6.000000,0.000000,13.900000,2.600000,1.790000
max,6.000000,57.000000,9.000000,9991.000000,1.000000,43.000000,190.000000,109.000000,44.000000,44.000000,...,26.000000,12.000000,19.000000,23.000000,33.000000,20.000000,12.000000,16.200000,3.700000,3.510000


In [ ]:
if df is not None:
    col_name = 'Curricular units 2nd sem (approved)'
    min_aprobadas = df[col_name].min()
    max_aprobadas = df[col_name].max()
    media_aprobadas = df[col_name].mean()

    print(f"Análisis de materias aprobadas (2do semestre):")
    print(f"- Mínimo: {min_aprobadas}")
    print(f"- Máximo: {max_aprobadas}")
    print(f"- Promedio: {media_aprobadas:.2f}")

    # Mostrar la distribución de los valores
    display(df[col_name].value_counts().sort_index().to_frame(name='Cantidad de Estudiantes'))

## 3. Data Preparation

En esta etapa prepararemos los datos para el modelo:
1.  Mapeo de la variable `Target` a valores numéricos (Riesgo).
2.  Manejo de valores nulos.
3.  Codificación de variables categóricas.
4.  Escalado de datos.
5.  División en conjuntos de entrenamiento y prueba.

In [7]:
# 1. Convertir la variable Target a valores numéricos de riesgo
target_map = {'Dropout': 1.0, 'Enrolled': 0.5, 'Graduate': 0.0}
df['Target_Risk'] = df['Target'].map(target_map)

# Convertir la columna 'Curricular units 2nd sem (grade)' a numérica antes de manejar nulos
# Esto es necesario porque el escalador requiere valores numéricos y esta columna es de tipo 'object'.
# 'errors="coerce"' convertirá cualquier valor que no se pueda transformar en número a NaN.
df['Curricular units 2nd sem (grade)'] = pd.to_numeric(df['Curricular units 2nd sem (grade)'], errors='coerce')

# 2. Manejo de nulos (si los hubiera). Ahora también manejará los NaNs introducidos por la conversión.
df = df.fillna(df.median(numeric_only=True))

# 3. Selección de características y Target
# Usaremos variables clave: Edad, Unidades curriculares, Género, Becas, Deudas
features = ['Age at enrollment', 'Curricular units 2nd sem (approved)',
            'Curricular units 2nd sem (grade)', 'Gender',
            'Scholarship holder', 'Debtor', 'Tuition fees up to date']

X = df[features]
y = df['Target_Risk']

# 4. División de datos (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 5. Escalado de datos
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Preparación de datos completada.")

Preparación de datos completada.


## 4. Modelado (Regresión Múltiple / Perceptrón Lineal)

Entrenaremos un modelo de regresión lineal, que puede interpretarse como un perceptrón de una sola capa con función de activación lineal para predecir el riesgo.

In [8]:
# Crear y entrenar el modelo
model = LinearRegression()
model.fit(X_train_scaled, y_train)

# Evaluación
y_pred = model.predict(X_test_scaled)
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"R² Score: {r2:.4f}")
print(f"Mean Squared Error: {mse:.4f}")
print("\nCoeficientes del modelo:")
for feat, coef in zip(features, model.coef_):
    print(f"{feat}: {coef:.4f}")

R² Score: 0.4608
Mean Squared Error: 0.1100

Coeficientes del modelo:
Age at enrollment: 0.0473
Curricular units 2nd sem (approved): -0.2288
Curricular units 2nd sem (grade): -0.0032
Gender: 0.0166
Scholarship holder: -0.0588
Debtor: 0.0302
Tuition fees up to date: -0.0834


## 5. Predicción con Datos Manuales

Esta función permite ingresar los datos de un estudiante para calcular su nivel de riesgo de deserción.

In [9]:
def predecir_riesgo():
    print("--- Simulador de Riesgo de Deserción ---")
    try:
        # Entradas basadas en las variables del modelo
        edad = float(input("1. Edad del estudiante al inscribirse: "))
        aprobados = float(input("2. Cantidad de materias aprobadas en el 2do semestre: "))
        nota = float(input("3. Promedio académico obtenido en el 2do semestre (0-20): "))
        genero = int(input("4. Género (1=Masculino, 0=Femenino): "))
        beca = int(input("5. ¿Es beneficiario de alguna beca? (1=Sí, 0=No): "))
        deudor = int(input("6. ¿Tiene deudas pendientes con la universidad? (1=Sí, 0=No): "))
        pagos = int(input("7. ¿Tiene los pagos de matrícula al día? (1=Sí, 0=No): "))

        # Crear array con los nombres de columnas para evitar el Warning de sklearn
        datos = pd.DataFrame([[edad, aprobados, nota, genero, beca, deudor, pagos]], columns=features)
        datos_scaled = scaler.transform(datos)

        # Predicción
        riesgo = model.predict(datos_scaled)[0]
        riesgo = max(0, min(1, riesgo)) # Normalizar entre 0 y 1

        print(f"\n--- RESULTADO DEL ANÁLISIS ---")
        print(f"Probabilidad de Deserción: {riesgo:.2%}")

        if riesgo > 0.7:
            print("NIVEL DE RIESGO: CRÍTICO (Alto)")
        elif riesgo > 0.4:
            print("NIVEL DE RIESGO: MODERADO (Medio)")
        else:
            print("NIVEL DE RIESGO: ESTABLE (Bajo)")

    except ValueError:
        print("Error: Por favor ingresa valores numéricos válidos (números para edad/notas y 1 o 0 para opciones).")

# Ejecutar la función de predicción
predecir_riesgo()

--- Simulador de Riesgo de Deserción ---
1. Edad del estudiante al inscribirse: 18
2. Cantidad de materias aprobadas en el 2do semestre: 4
3. Promedio académico obtenido en el 2do semestre (0-20): 17
4. Género (1=Masculino, 0=Femenino): 1
5. ¿Es beneficiario de alguna beca? (1=Sí, 0=No): 0
6. ¿Tiene deudas pendientes con la universidad? (1=Sí, 0=No): 0
7. ¿Tiene los pagos de matrícula al día? (1=Sí, 0=No): 1

--- RESULTADO DEL ANÁLISIS ---
Probabilidad de Deserción: 41.73%
NIVEL DE RIESGO: MODERADO (Medio)
